In [2]:
import pandas as pd
import glob
import os
import re

CHASE_CREDIT_DIR    = '../original_csv/credit_cards/chase'
ALLIANT_CREDIT_DIR  = '../original_csv/credit_cards/alliant'
ALLIANT_CHECKING_DIR = '../original_csv/checking/alliant'
CHASE_CHECKING_DIR  = '../original_csv/checking/chase'

def csv_files(directory):
    return (
        glob.glob(os.path.join(directory, '*.csv')) +
        glob.glob(os.path.join(directory, '*.CSV'))
    )

def parse_alliant_amount(val):
    val = str(val).strip()
    negative = val.startswith('(')
    val = re.sub(r'[\$,()\s]', '', val)
    return -float(val) if negative else float(val)

# --- Chase credit card ---
chase_credit_frames = []
for f in csv_files(CHASE_CREDIT_DIR):
    df = pd.read_csv(f)
    df = df[['Transaction Date', 'Description', 'Amount', 'Category']].copy()
    df.rename(columns={'Transaction Date': 'date', 'Description': 'description',
                       'Amount': 'amount', 'Category': 'category'}, inplace=True)
    df['account'] = 'chase_credit'
    chase_credit_frames.append(df)

chase_credit = pd.concat(chase_credit_frames, ignore_index=True) if chase_credit_frames else pd.DataFrame()
if not chase_credit.empty:
    chase_credit['date'] = pd.to_datetime(chase_credit['date'])
    # Chase: negative = expenditure, positive = deposit — already normalized

# --- Alliant credit card ---
alliant_credit_frames = []
for f in csv_files(ALLIANT_CREDIT_DIR):
    df = pd.read_csv(f)
    df = df[['Date', 'Description', 'Amount']].copy()
    df.rename(columns={'Date': 'date', 'Description': 'description'}, inplace=True)
    df['amount'] = df['Amount'].apply(parse_alliant_amount)
    df.drop(columns=['Amount'], inplace=True)
    df['category'] = ''
    df['account'] = 'alliant_credit'
    alliant_credit_frames.append(df)

alliant_credit = pd.concat(alliant_credit_frames, ignore_index=True) if alliant_credit_frames else pd.DataFrame()
if not alliant_credit.empty:
    alliant_credit['date'] = pd.to_datetime(alliant_credit['date'])
    # Alliant credit: positive = charge, parens = payment
    # Negate so negative = expenditure, positive = deposit (consistent with Chase)
    alliant_credit['amount'] = -alliant_credit['amount']

# --- Alliant checking ---
alliant_checking_frames = []
for f in csv_files(ALLIANT_CHECKING_DIR):
    df = pd.read_csv(f)
    df = df[['Date', 'Description', 'Amount']].copy()
    df.rename(columns={'Date': 'date', 'Description': 'description'}, inplace=True)
    df['amount'] = df['Amount'].apply(parse_alliant_amount)
    df.drop(columns=['Amount'], inplace=True)
    df['category'] = ''
    df['account'] = 'alliant_checking'
    alliant_checking_frames.append(df)

alliant_checking = pd.concat(alliant_checking_frames, ignore_index=True) if alliant_checking_frames else pd.DataFrame()
if not alliant_checking.empty:
    alliant_checking['date'] = pd.to_datetime(alliant_checking['date'])
    # Alliant checking: positive = deposit, parens/negative = withdrawal — already normalized

# --- Chase checking ---
chase_checking_frames = []
for f in csv_files(CHASE_CHECKING_DIR):
    df = pd.read_csv(f)
    df = df[['Transaction Date', 'Description', 'Amount']].copy()
    df.rename(columns={'Transaction Date': 'date', 'Description': 'description',
                       'Amount': 'amount'}, inplace=True)
    df['category'] = ''
    df['account'] = 'chase_checking'
    chase_checking_frames.append(df)

chase_checking = pd.concat(chase_checking_frames, ignore_index=True) if chase_checking_frames else pd.DataFrame()
if not chase_checking.empty:
    chase_checking['date'] = pd.to_datetime(chase_checking['date'])

# --- Combine all ---
combined = pd.concat(
    [df for df in [chase_credit, alliant_credit, alliant_checking, chase_checking] if not df.empty],
    ignore_index=True
)
combined['subcategory'] = pd.NA
combined['merchant'] = pd.NA
combined['ignore'] = False
combined['necessity'] = pd.NA
combined = combined[['date', 'description', 'amount', 'account', 'category',
                     'subcategory', 'merchant', 'ignore', 'necessity']]
combined.sort_values('date', inplace=True)
combined.reset_index(drop=True, inplace=True)

# --- Split and save ---
expenditures = combined[combined['amount'] < 0].copy()

deposits = combined[combined['amount'] > 0][['date', 'description', 'amount', 'account', 'category']].copy()

expenditures.to_csv('../expenditures.csv')
deposits.to_csv('../deposits.csv')

print(f"Expenditures: {len(expenditures)} rows → expenditures.csv")
print(f"Deposits:     {len(deposits)} rows → deposits.csv")

Expenditures: 71 rows → expenditures.csv
Deposits:     15 rows → deposits.csv
